# Creating a Master Bad Pixel File


**Author: Dave Strickland**

This notebook illustrates how to generate a master bad pixel files given a master dark. The master bad pixel file is normally used as part of the calibration process (e.g. `ApCalibrate` or `ap_calibrate.py` from the command line), but can also be applied to calibrated files when you do not have the raw images. You must have a master dark frame, the longer the exposuer time the better. Master bias files can be used, but will fail to pick up hot pixels where the charge increases with time.

The python environment used corresponds to a miniconda emvironment `ap-env.yml`.

The example darks used are iTelescope T24 calibration files from 
Note that these files are not provided with this package. However the notebook should work with your own files if you specify a valid fits-file containing directory at the prompt below.

The notebook demonstrates processing using existing master dark files in the first section, followed by generation of master calibration files from raw dark, bias and flat field files in a later section.

Although some `matplotlib` images of the dark and bad pixel files are shown, much of the by-hand checking is done using [SAOImage ds9](https://sites.google.com/cfa.harvard.edu/saoimageds9). (Which I highly recommend.) 

## Notebook environment setup

In [30]:
import os
import pathlib
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time
import math
import subprocess
import shlex

from ccdproc import ImageFileCollection
from astropy.table import Table
from astropy import wcs
import astropy
from astropy.io import fits

import AstroPhotography as ap

In [31]:
def print_module_version(mod):
    """
    Convenience function to pretty-print Python module mod version.

    Arguments:
        mod {module} -- Imported Python module

    Returns:
        str -- module name and version
    """
    print(f"Using module {mod.__name__:30s}  version: {mod.__version__}")
    return

In [32]:
# Get Version information
print(f'Python version: {sys.version}')
print_module_version(np)
print_module_version(matplotlib)
print_module_version(astropy)
print_module_version(ap)

Python version: 3.10.13 | packaged by conda-forge | (main, Dec 23 2023, 15:36:39) [GCC 12.3.0]
Using module numpy                           version: 1.26.3
Using module matplotlib                      version: 3.8.2
Using module astropy                         version: 6.0.0
Using module AstroPhotography                version: 0.5.2


In [33]:
# Enable inline plotting for graphics
# %matplotlib inline
# Set default figure size to be larger
# this may only work in matplotlib 2.0+!
from IPython.core.interactiveshell import InteractiveShell
matplotlib.rcParams['figure.figsize'] = [10.0, 6.0]
# Enable multiple outputs from jupyter cells
InteractiveShell.ast_node_interactivity = "all"

# Processing from Master Dark Files

## What files do we have to work with?

Let us change the working directory from the location of this notebook to a directory holding some iTelescope dark files. In my case this is the most recent set of calibration data for [iTelescope T24](https://support.itelescope.net/support/solutions/articles/231907-telescope-24). Dark and bias handling for T24 and T31 are more complicated than most iTelescope telescopes as they require [Residual Bulk Image (RBI) flushing](https://www.gxccd.com/art?id=418&lang=409), but for the purposes of generating bad pixel files we can ignore the RBI issue.

**Note:** I have renamed the iTelescope files to replace whitespace characters with underscores.

In [34]:
default_path = '/old_lnx/home/dks/Downloads/iTelescopeScratch/calibration-library/T24/Raw/2021_DEC/'
print(f'Enter the full path to the directory containing the master dark calibration files, or return to accept {default_path}')
wdir = input('Dark file set path:').strip() or default_path
try:
    os.chdir(wdir.strip())
    print('Switched directory to ' + os.getcwd())
except:
    print(f'Error, os.chdir threw an exception changing to {wdir}')
    print('Check that the path you supplied is a valid filesystem path.')
    raise

Enter the full path to the directory containing the master dark calibration files, or return to accept /old_lnx/home/dks/Downloads/iTelescopeScratch/calibration-library/T24/Raw/2021_DEC/


Dark file set path: 


Switched directory to /old_lnx/home/dks/Downloads/iTelescopeScratch/calibration-library/T24/Raw/2021_DEC


We now want to select some files of different binning, temperature, and the greatest exposure time we can find.

Listing the fits files from the shell I get:
```bash
find . -name "*900s*"
./Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_1_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
./Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_2_1528x1528_Bin2x2_Temp-25C_ExpTime900s.fit
./Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_12_1528x1528_Bin2x2_Temp-25C_ExpTime900s.fit
./Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
./Darks/Master_Dark_12_1528x1528_Bin2x2_Temp-25C_ExpTime900s.fit
./Darks/Master_Dark_6_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
```

In this case we want the 900 second exposure files, so a valid pattern that picks up the original files is `Master_Dark*900s.fit*`

In [35]:
default_pattern = 'Master_Dark*900s.fit*'
print(f'Default file pattern for input files: {default_pattern}')
msg = 'Enter new input file pattern (or return to accept default pattern)'
file_pattern = input(msg).strip() or default_pattern
print(f'Using "{file_pattern}" as the input file pattern.')

Default file pattern for input files: Master_Dark*900s.fit*


Enter new input file pattern (or return to accept default pattern) 


Using "Master_Dark*900s.fit*" as the input file pattern.


In [36]:

p_dir = pathlib.Path(r'./')
flist = list(sorted(p_dir.rglob(file_pattern)))
print(f'{len(flist)} files match pattern "{file_pattern}" in current directory.')
for fpath in flist:
    print(f'  {fpath}')

7 files match pattern "Master_Dark*900s.fit*" in current directory.
  Darks/Master_Dark_12_1528x1528_Bin2x2_Temp-25C_ExpTime900s.fit
  Darks/Master_Dark_6_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
  Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
  Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_12_1528x1528_Bin2x2_Temp-25C_ExpTime900s.fit
  Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_1_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
  Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_2_1528x1528_Bin2x2_Temp-25C_ExpTime900s.fit
  Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit


In [37]:
# Look at the files in ds9. Note the annoying spaces cause trouble...
import shutil
if shutil.which("ds9") is not None:
    print('Calling shell to execute ds9...')
    !ds9 -cmap viridis -asinh -scale mode 99.5 -zoom 0.125 $(find . -name "$default_pattern" | xargs)
else:
    print('No ds9 was found in your $PATH')

Calling shell to execute ds9...


### Command line usage

To create the initial bad pixel file we could use `ap_find_badpix.py` on the command line, or the class `ApFindBadPixels`. Based on the `ds9` images I'll just use the `Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit` file.

```bash
ln -s Darks_RBI_flood_5_x_5sec/Darks/Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
python3 ~/git/AstroPhotography/AstroPhotography/scripts/ap_find_badpix.py -l DEBUG \
    Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit \
    Master_Badpix_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit
```

Normally this would be followed by `ap_auto_badcol.py` (command line) or `ApAutoBadcols`. Copy the user badpix YaML template from `~/git/AstroPhotography/etc/user_badpixels.yml` to  `t24_user_badpixels_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.yml` and insert the output by hand into the YaML file.

```bash
python3 ~/git/AstroPhotography/AstroPhotography/scripts/ap_auto_badcol.py -l DEBUG Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit

# update the master bad pixel file
python3 ~/git/AstroPhotography/AstroPhotography/scripts/ap_find_badpix.py -l DEBUG \
    Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit \
    Master_Badpix_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit \
    --user_badpix t24_user_badpixels_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.yml
```

### Python interface

In [38]:
masterdark = 'Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit'
badpixfile = masterdark.replace('Dark', 'Badpix')
print(f'For master dark {masterdark}, output bad pixel file is {badpixfile}')

# Have not got a file yet
userbadpix = None

# Defaults, don't have to actually use these
loglevel = 'INFO'
sigma    = 4.0
mkbadpix = ap.ApFindBadPixels(masterdark)
        
if userbadpix is not None:
    mkbadpix.add_user_badpix(userbadpix)
        
# Write final bad pixels mask.
mkbadpix.write_mask(p_outbadpix)

For master dark Master_Dark_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit, output bad pixel file is Master_Badpix_11_3056x3056_Bin1x1_Temp-25C_ExpTime900s.fit


TypeError: ApFindBadPixels.__init__() missing 2 required positional arguments: 'sigma' and 'loglevel'